# 05 — v2 & ML Comparison

Compare v1 baseline, v2 improvements, and ML autoencoder reconstruction.

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from src.data_loader.loader import load_signal
from src.generate_synthetic import create_synthetic_dataset
from src.preprocessing.noise import apply_all_noise
from src.preprocessing.filtering import butterworth_lowpass
from src.compression.fft_compression import compress_fft, decompress_fft
from src.v2.compression.adaptive_fft import compress_adaptive_fft, decompress_adaptive_fft
from src.v2.compression.hybrid import compress_hybrid, decompress_hybrid
from src.v2.denoising.pipeline import denoise_multistage
from src.metrics.mse import mse
from src.metrics.snr import snr_db
from src.metrics.compression_ratio import compression_ratio
from src.visualization.plots import plot_comparison
%matplotlib inline

In [ ]:
path = create_synthetic_dataset()
time, signal, meta = load_signal(path)
fs = meta['sampling_frequency']
noisy, _ = apply_all_noise(signal, seed=42)

# v1 path
v1_filtered = butterworth_lowpass(noisy, fs, 15.0)
v1_comp = compress_fft(v1_filtered, 0.1)
v1_recon = decompress_fft(v1_comp)

# v2 path
v2_filtered, _ = denoise_multistage(noisy, fs)
v2_comp = compress_adaptive_fft(v2_filtered, 0.90)
v2_recon = decompress_adaptive_fft(v2_comp)

# v2 hybrid
hybrid_comp = compress_hybrid(v2_filtered, 0.85, 0.15)
hybrid_recon = decompress_hybrid(hybrid_comp)

print(f"{'Method':<25} {'SNR':>8} {'MSE':>10} {'Ratio':>8}")
for name, orig, recon, comp in [
    ('v1 FFT 10%', v1_filtered, v1_recon, v1_comp),
    ('v2 Adaptive FFT', v2_filtered, v2_recon, v2_comp),
    ('v2 Hybrid', v2_filtered, hybrid_recon, hybrid_comp),
]:
    print(f"{name:<25} {snr_db(orig,recon):>7.1f} {mse(orig,recon):>10.6f} {compression_ratio(orig,comp):>7.1f}x")

In [ ]:
try:
    from src.ml.autoencoder import compress_ml, decompress_ml
    ml_comp = compress_ml(v2_filtered, latent_dim=32, epochs=30)
    ml_recon = decompress_ml(ml_comp)
    print(f"ML Autoencoder: SNR={snr_db(v2_filtered, ml_recon):.1f} dB, MSE={mse(v2_filtered, ml_recon):.6f}")
    plot_comparison(time, v2_filtered, ml_recon, 'Denoised', 'ML Reconstructed', 'ML Autoencoder');
except ImportError:
    print('PyTorch not installed — skip ML section')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for ax, recon, title in zip(axes, [v1_recon, v2_recon, hybrid_recon],
                            ['v1 FFT', 'v2 Adaptive FFT', 'v2 Hybrid']):
    ax.plot(time, v2_filtered, alpha=0.3, label='Denoised')
    ax.plot(time[:len(recon)], recon, label=title)
    ax.legend(); ax.set_title(title); ax.grid(True, alpha=0.3)
plt.xlabel('Time (s)'); plt.tight_layout();